# RAG 第 10 课：A/B 对比与实验设计

前一课我们已经有了一个完整的教学版 RAG pipeline：
- `query rewrite`
- `retrieve`
- `rerank`
- `generate`
- `evaluate`

但真实工作里，真正高频的问题不是“能不能搭起来”，而是：

- 我改了 query rewrite，真的变好了吗？
- 我换了 reranker，值得上线吗？
- 这次提升是整体提升，还是只对某类 query 有帮助？

所以这一课我们来学：

**怎么做一个最小可运行的 A/B 对比实验。**

In [ ]:
# 先清理 GPU 缓存
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 实验背景

这一课我们设计两个版本：

- `System A`：不做 query rewrite
- `System B`：做一个简单的 query rewrite

然后我们在同一份 eval set 上比较：
- 检索是否更准
- 最终答案是否更准
- 哪些 query 受益最大

In [ ]:
import re
from collections import Counter, defaultdict

kb = [
    'E401 means authentication failed because the API token is invalid or expired.',
    'Reset your password by opening Settings, then Security, then Reset Password.',
    'Chunking splits long documents into smaller passages to improve retrieval focus.',
    'Embeddings convert text into vectors so semantic similarity can be measured.',
    'Hybrid search combines lexical matching and dense retrieval scores.',
    'Reranking reorders retrieved candidates so the strongest evidence appears first.',
    'Billing reports can be exported from the Reports page as CSV files.',
]

dataset = [
    {
        'query': 'Why is my token being rejected with auth failure?',
        'gold_doc_id': 0,
        'reference_answer': 'E401 means authentication failed because the token is invalid or expired.',
        'category': 'paraphrase',
    },
    {
        'query': 'How do I update my login credentials?',
        'gold_doc_id': 1,
        'reference_answer': 'Open Settings, go to Security, and choose Reset Password.',
        'category': 'paraphrase',
    },
    {
        'query': 'Why break long docs into passages in RAG?',
        'gold_doc_id': 2,
        'reference_answer': 'Chunking splits long documents into smaller passages so retrieval becomes more focused.',
        'category': 'concept',
    },
    {
        'query': 'Can I save billing statements as a spreadsheet?',
        'gold_doc_id': 6,
        'reference_answer': 'You can export billing reports from the Reports page as CSV files.',
        'category': 'product_howto',
    },
    {
        'query': 'What does E401 mean?',
        'gold_doc_id': 0,
        'reference_answer': 'E401 means authentication failed because the token is invalid or expired.',
        'category': 'direct_fact',
    },
]

## 2. 公共组件

A 和 B 两个系统会共享这些底层工具。
这样我们就能更干净地比较“只改了 query rewrite”到底带来了什么影响。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('ed') and len(token) > 4:
        token = token[:-2]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(tokenize(text))


def lexical_score(query, text):
    q_tokens = tokenize(query)
    d_tokens = set(tokenize(text))
    score = 0.0
    for token in q_tokens:
        if token in d_tokens:
            score += 1.0
            if any(ch.isdigit() for ch in token):
                score += 1.5
    return score


def retrieve(query, top_k=3):
    rows = []
    for doc_id, doc in enumerate(kb):
        rows.append({'doc_id': doc_id, 'score': lexical_score(query, doc), 'text': doc})
    rows.sort(key=lambda x: x['score'], reverse=True)
    return rows[:top_k]


def rerank(query, candidates):
    q_tokens = set(tokenize(query))
    reranked = []
    for row in candidates:
        doc_tokens = set(tokenize(row['text']))
        overlap = len(q_tokens & doc_tokens)
        bonus = 0.0
        lowered = row['text'].lower()

        if 'csv' in query.lower() and 'csv' in lowered:
            bonus += 1.0
        if 'authentication' in query.lower() and 'authentication failed' in lowered:
            bonus += 1.0
        if 'reset password' in query.lower() and 'reset password' in lowered:
            bonus += 1.0

        reranked.append({**row, 'rerank_score': overlap + bonus + 0.1 * row['score']})

    reranked.sort(key=lambda x: x['rerank_score'], reverse=True)
    return reranked


def generate_answer(original_query, context):
    q = original_query.lower()
    c = context.lower()

    if 'e401' in q:
        if 'e401' in c:
            return 'E401 means authentication failed because the token is invalid or expired.'
        return 'E401 is related to a timeout issue.'

    if 'password' in q:
        if 'reset password' in c:
            return 'Open Settings, go to Security, and choose Reset Password.'
        return 'You need to contact support to change your password.'

    if 'chunking' in q:
        if 'chunking' in c:
            return 'Chunking splits long documents into smaller passages so retrieval becomes more focused.'
        return 'Chunking mainly reduces storage size.'

    if 'billing reports' in q or 'report' in q:
        if 'csv' in c or 'reports page' in c:
            return 'You can export billing reports from the Reports page as CSV files.'
        return 'Billing reports cannot be exported.'

    return 'I do not know.'


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 3. 两个系统版本

`System A` 不做 rewrite，直接检索。  
`System B` 先 rewrite，再检索。

In [ ]:
def identity_rewrite(query):
    return query.lower()


def simple_rewrite(query):
    rewritten = query.lower()
    replacements = {
        'auth failure': 'authentication failed e401',
        'token being rejected': 'token invalid or expired',
        'update my login credentials': 'reset password',
        'break long docs into passages': 'chunking improve retrieval focus',
        'billing statements': 'billing reports',
        'spreadsheet': 'csv',
    }
    for src, dst in replacements.items():
        rewritten = rewritten.replace(src, dst)
    return rewritten


def run_pipeline(query, rewrite_fn):
    rewritten_query = rewrite_fn(query)
    retrieved = retrieve(rewritten_query, top_k=3)
    reranked = rerank(rewritten_query, retrieved)
    chosen = reranked[0]
    answer = generate_answer(query, chosen['text'])
    return {
        'rewritten_query': rewritten_query,
        'retrieved': retrieved,
        'reranked': reranked,
        'chosen_doc_id': chosen['doc_id'],
        'answer': answer,
    }

In [ ]:
example_query = dataset[0]['query']
result_a = run_pipeline(example_query, identity_rewrite)
result_b = run_pipeline(example_query, simple_rewrite)

print('query:', example_query)
print('\nSystem A rewritten_query:', result_a['rewritten_query'])
print('System A chosen_doc_id:', result_a['chosen_doc_id'])
print('System A answer:', result_a['answer'])

print('\nSystem B rewritten_query:', result_b['rewritten_query'])
print('System B chosen_doc_id:', result_b['chosen_doc_id'])
print('System B answer:', result_b['answer'])

## 4. 在同一份 eval set 上做 A/B 对比

这是最关键的原则：

**A 和 B 一定要跑在同一份数据上。**

不然你看到的差异，可能只是数据不同，而不是系统真的变好了。

In [ ]:
def evaluate_system(dataset, rewrite_fn, system_name):
    rows = []
    retrieval_scores = []
    em_scores = []
    f1_scores = []

    for item in dataset:
        result = run_pipeline(item['query'], rewrite_fn)
        retrieval_hit = 1.0 if result['chosen_doc_id'] == item['gold_doc_id'] else 0.0
        em = exact_match(result['answer'], item['reference_answer'])
        f1 = token_f1(result['answer'], item['reference_answer'])

        retrieval_scores.append(retrieval_hit)
        em_scores.append(em)
        f1_scores.append(f1)

        rows.append({
            'system': system_name,
            'query': item['query'],
            'category': item['category'],
            'gold_doc_id': item['gold_doc_id'],
            'chosen_doc_id': result['chosen_doc_id'],
            'retrieval_hit': retrieval_hit,
            'exact_match': em,
            'token_f1': f1,
            'rewritten_query': result['rewritten_query'],
            'answer': result['answer'],
        })

    summary = {
        'system': system_name,
        'RetrievalHit@1': sum(retrieval_scores) / len(retrieval_scores),
        'ExactMatch': sum(em_scores) / len(em_scores),
        'TokenF1': sum(f1_scores) / len(f1_scores),
    }
    return summary, rows

In [ ]:
summary_a, rows_a = evaluate_system(dataset, identity_rewrite, 'System A')
summary_b, rows_b = evaluate_system(dataset, simple_rewrite, 'System B')

print(summary_a)
print(summary_b)

## 5. 不只是看总分，还要看 delta

真实项目里我们经常会直接看：

`B - A`

也就是新版本相对旧版本到底提升了多少。

In [ ]:
delta = {
    'RetrievalHit@1_delta': summary_b['RetrievalHit@1'] - summary_a['RetrievalHit@1'],
    'ExactMatch_delta': summary_b['ExactMatch'] - summary_a['ExactMatch'],
    'TokenF1_delta': summary_b['TokenF1'] - summary_a['TokenF1'],
}
print(delta)

## 6. 看哪些 query 真正受益了

如果只看平均分，你很容易错过一个事实：

- 有些 query 明显变好了
- 有些 query 没变化
- 有些 query 甚至可能变差了

所以我们要把 A/B 结果按 query 对齐后再比较。

In [ ]:
comparison_rows = []
for row_a, row_b in zip(rows_a, rows_b):
    comparison_rows.append({
        'query': row_a['query'],
        'category': row_a['category'],
        'A_doc': row_a['chosen_doc_id'],
        'B_doc': row_b['chosen_doc_id'],
        'A_hit': row_a['retrieval_hit'],
        'B_hit': row_b['retrieval_hit'],
        'A_em': row_a['exact_match'],
        'B_em': row_b['exact_match'],
        'A_rewrite': row_a['rewritten_query'],
        'B_rewrite': row_b['rewritten_query'],
    })

for row in comparison_rows:
    print('query:', row['query'])
    print('category:', row['category'])
    print('A_doc / B_doc:', row['A_doc'], '/', row['B_doc'])
    print('A_hit / B_hit:', row['A_hit'], '/', row['B_hit'])
    print('A_em  / B_em :', row['A_em'], '/', row['B_em'])
    print('A_rewrite:', row['A_rewrite'])
    print('B_rewrite:', row['B_rewrite'])
    print('-' * 90)

## 7. 再进一步：按 category 看提升

这一步特别像真实项目里的分析方式。
因为很多改动并不是“全局平均变好”，而是：

**对某一类 query 很有帮助。**

In [ ]:
def summarize_by_category(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row['category']].append(row)

    summary = []
    for category, items in grouped.items():
        summary.append({
            'category': category,
            'count': len(items),
            'RetrievalHit@1': sum(x['retrieval_hit'] for x in items) / len(items),
            'ExactMatch': sum(x['exact_match'] for x in items) / len(items),
            'TokenF1': sum(x['token_f1'] for x in items) / len(items),
        })
    return sorted(summary, key=lambda x: x['category'])


category_a = summarize_by_category(rows_a)
category_b = summarize_by_category(rows_b)

print('System A by category:')
for row in category_a:
    print(row)

print('\nSystem B by category:')
for row in category_b:
    print(row)

## 8. 这一课最想让你建立的直觉

A/B 对比最重要的不是“写个表格”，而是遵守几个基本原则：

1. 用同一份 eval set 比
2. 尽量一次只改一个变量
3. 不只看平均分，也看失败样本
4. 按 query 类型切分看效果

这样你才能知道：
- 这次改动到底有没有价值
- 它帮助的是哪一类问题
- 它有没有带来副作用

## 9. 本课小结

这节课你要带走的核心是：

1. RAG 优化不能只靠感觉，要做 A/B 对比。
2. A/B 的关键是：同数据、少变量、看 delta、看失败样本。
3. 按 category 分析，往往比只看总分更有价值。

下一课最自然的衔接就是：
**RAG 里的常见失败模式与调优思路**，也就是更系统化地总结“问题出在哪、该怎么改”。